# Biomarker discovery and modeling

## Introduction and objectives

This analysis aims to identify biomarkers for a panel that could eventually
be measured by ddPCR and a multiplex protein immunoassay.

The available data include:

- 350 samples from 300 patients
- Methylation region measurements with annotations (genomic locations, nearest genes, GC content, and repeat overlap)
- Protein concentrations (pg/mL), which are unavailable for some patients
- Genome-wide fragmentomics data for all patients

The proposed final panel can include:

- Up to 5 methylation targets measured by ddPCR
- Up to 3 protein markers measured by multiplex immunoassay

We therefore aim to identify methylation and protein markers suitable for this
panel.

## Strategy and decisions

The analysis follows this plan:

- Assess data quality by examining distributions, potential confounders, and missing values in tables and plots.
- Define imputation strategies where needed.
- Use PCA plots for methylation, fragmentomics, and protein data to identify potential outliers or mislabeled samples (for example, a cancer sample labeled as normal).
- Exclude samples with more than 20% missing values within a modality.
- Exclude features with more than 20% missing values within a modality.
- Keep only the earliest draw from patients with repeated samples to avoid giving those patients extra weight in modeling.
- Identify site and plate effects using control samples, then apply the estimated corrections to all samples.
- Build cross-validated LASSO models using (1) methylation and protein, (2) assay-design-filtered
methylation and protein, and (3) methylation, protein, and fragmentomics.
Every imputation, batch correction, and scaling rule is learned again inside
each training fold.
- Consider selection frequency across repeated CV, early-stage sensitiviy, and technical robustness (including missingness) when choosing the final markers.
- Examine final marker properties, including GC content, nearby genes, and differences between early- or late-stage cancers and controls.


## Setup

Set `PROJECT_ROOT`, `INPUT_DIR`, and `OUTPUT_DIR` below if the notebook is
run from a different working directory. The default experiment setup is: five repeats, five outer folds, three inner folds. A full run can take substantial time because it fits nested models three times.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "candidate_data").exists() else CURRENT_DIR.parent
if not (PROJECT_ROOT / "candidate_data").exists():
    raise FileNotFoundError("Set PROJECT_ROOT to the assignment directory.")
INPUT_DIR = PROJECT_ROOT / "candidate_data"
OUTPUT_DIR = PROJECT_ROOT / "report_artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPEATS = 5
OUTER_FOLDS = 5
INNER_FOLDS = 3
SEED = 20260927
LAMBDA_FRACTIONS = (0.9, 0.65, 0.45, 0.3, 0.2, 0.13, 0.08, 0.05, 0.03, 0.015)
MOLECULAR_CAPS = {"methylation": 5, "protein": 3}
FRAGMENTOMICS_CAPS = {**MOLECULAR_CAPS, "fragmentomics": 2}
# print(f"Input: {INPUT_DIR}\nOutput: {OUTPUT_DIR}")

## Results

### QC: earliest draws and cohort composition

Fifty patients have two draws. Keeping the earliest avoids giving these
patients extra weight during training; later draws can be examined separately.
The tables and plots below describe the earliest-draw cohort. "Control"
denotes patients without cancer.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import f as f_distribution
from scipy.special import expit
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


def read_inputs(input_dir: Path) -> dict[str, pd.DataFrame]:
    """Read and validate the five source tables.

    Parameters
    ----------
    input_dir : Path
        Directory containing the assignment CSV files.

    Returns
    -------
    dict[str, pandas.DataFrame]
        Source tables keyed by file stem.
    """
    names = ("samples", "methylation", "fragmentomics", "protein", "region_annotation")
    tables = {name: pd.read_csv(input_dir / f"{name}.csv") for name in names}
    for name in names[:-1]:
        if tables[name]["sample_id"].duplicated().any():
            raise ValueError(f"Duplicate sample IDs in {name}.")
    if tables["region_annotation"]["region_id"].duplicated().any():
        raise ValueError("Duplicate region IDs.")
    tables["samples"]["draw_date"] = pd.to_datetime(tables["samples"]["draw_date"])
    if tables["samples"]["draw_date"].isna().any():
        raise ValueError("Missing or invalid collection dates.")
    return tables


def build_earliest_draw_cohort(
    samples: pd.DataFrame, assays: dict[str, pd.DataFrame]
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """Keep one draw per patient and align every assay to that draw.

    Parameters
    ----------
    samples : pandas.DataFrame
        Sample metadata with patient, sample, and collection-date columns.
    assays : dict[str, pandas.DataFrame]
        Assay tables keyed by modality.

    Returns
    -------
    tuple[pandas.DataFrame, dict[str, pandas.DataFrame]]
        Patient metadata and sample-aligned assay tables.
    """
    primary = (samples.sort_values(["patient_id", "draw_date", "sample_id"])
               .drop_duplicates("patient_id").reset_index(drop=True).copy())
    primary["stage_group"] = np.where(primary["cancer_status"].eq("non_cancer"),
                                       "Control", primary["stage"])
    primary["stage_group"] = pd.Categorical(
        primary["stage_group"], categories=["Control", "I", "II", "III", "IV"], ordered=True
    )
    primary["y"] = primary["cancer_status"].eq("cancer").astype(int)
    primary["age_group"] = pd.cut(
        primary["age"], bins=[-np.inf, 39, 49, 59, 69, np.inf],
        labels=["<40", "40–49", "50–59", "60–69", "70+"]
    )
    aligned = {}
    for modality, assay in assays.items():
        aligned[modality] = assay.set_index("sample_id").reindex(primary["sample_id"]).reset_index()
        # Reindex introduces the requested ID even when its assay row is absent.
        # Restore NA so structural assay absence remains distinct from blank cells.
        present = primary["sample_id"].isin(assay["sample_id"])
        aligned[modality].loc[~present, "sample_id"] = np.nan
    primary["plate_id"] = aligned["protein"]["plate_id"]
    primary["plate"] = primary["plate_id"].fillna("NoProtein")
    if primary["patient_id"].duplicated().any() or primary["stage_group"].isna().any():
        raise ValueError("Invalid earliest-draw cohort.")
    return primary, aligned


def assay_feature_columns(assay: pd.DataFrame) -> list[str]:
    """Return assay measurement columns, excluding sample and plate metadata.

    Parameters
    ----------
    assay : pandas.DataFrame
        One aligned assay table.

    Returns
    -------
    list[str]
        Measurement column names.
    """
    return [col for col in assay if col not in {"sample_id", "plate_id", "well"}]

The table below shows how many patients have repeated draws:

In [ ]:
tables = read_inputs(INPUT_DIR)
primary, assays_primary = build_earliest_draw_cohort(
    tables["samples"], {name: tables[name] for name in ("methylation", "fragmentomics", "protein")}
)
draw_count = tables["samples"].groupby("patient_id").size().value_counts().sort_index()
display(draw_count.rename_axis("Draws per patient").to_frame("Patients"))

The following table and plots shows how the biological factors (sex and age) are distributed among cancer stages, which shows that the samples are roughly evenly distributed among the two factors

In [ ]:
stage_sex_age = (primary.groupby(["stage_group", "sex", "age_group"], observed=True)
                 .size().rename("n_patients").reset_index())
display(stage_sex_age)
plot = sns.catplot(data=stage_sex_age, x="stage_group", y="n_patients",
                   hue="age_group", col="sex", kind="bar", height=4, aspect=1.3)
plot.set_axis_labels("Cancer stage", "Patients")
plt.show()

Next it also shows how the samples are distributed among batch variables: site and plate, where it shows that site A has more cancer samples than site B and C, where within each site, the distributions between plates are similar.

In [ ]:

stage_site_plate = (primary.groupby(["site", "plate", "stage_group"], observed=True)
                    .size().rename("n_patients").reset_index())
display(stage_site_plate)
plot = sns.catplot(data=stage_site_plate, x="stage_group", y="n_patients",
                   hue="plate", col="site", kind="bar", height=4, aspect=1)
plot.set_axis_labels("Cancer stage", "Patients")
plt.show()

The earliest-draw cohort has 300 patients. Site A has a larger cancer
proportion than sites B and C; site C has no protein assay rows. Plate PLT01
is restricted to site A and PLT03 to site B, while PLT02 spans both sites.

### QC: missing values

Next, we examine missing values at both the feature and sample levels.

- For each feature, calculate the fraction of samples with a missing value.
- For each sample, calculate the number of missing features within each modality, such as methylation.

Samples or features with excessive missingness can be removed; remaining missing
values can be imputed where needed.

- Missing methylation values reflect insufficient coverage, a technical cause rather than biological absence. We use median imputation because methylation values often cluster near 0 or 1, and mean imputation could produce misleading intermediate values.
- Missing protein values reflect concentrations below the lower limit of quantification (LLOQ). We therefore use half the LLOQ for imputation. Because the assignment does not provide LLOQs, we approximate each one with the smallest observed value for that protein.


In [ ]:
def summarize_missingness(
    assays: dict[str, pd.DataFrame], n_primary: int
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Summarize assay coverage and cell missingness among assayed patients.

    Parameters
    ----------
    assays : dict[str, pandas.DataFrame]
        Sample-aligned assay tables.
    n_primary : int
        Number of earliest-draw patients.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame, pandas.DataFrame]
        Coverage, feature-missingness, and sample-missingness tables.
    """
    coverage_rows, feature_rows, sample_rows = [], [], []
    for modality, assay in assays.items():
        measured = assay.loc[assay["sample_id"].notna()]
        features = assay_feature_columns(assay)
        missing = measured[features].isna()
        label = modality.capitalize()
        coverage_rows.append({"modality": label, "n_primary": n_primary,
                              "n_assayed": len(measured),
                              "n_without_assay_row": n_primary - len(measured)})
        for feature in features:
            feature_rows.append({"modality": label, "feature": feature,
                                 "n_assayed": len(measured),
                                 "n_missing": int(missing[feature].sum()),
                                 "missing_fraction": float(missing[feature].mean())})
        for sample_id, (_, row) in zip(measured["sample_id"], missing.iterrows()):
            sample_rows.append({"modality": label, "sample_id": sample_id,
                                "n_features": len(features), "n_missing": int(row.sum()),
                                "missing_fraction": float(row.mean())})
    return (pd.DataFrame(coverage_rows), pd.DataFrame(feature_rows),
            pd.DataFrame(sample_rows))


def filter_qc(
    cohort: pd.DataFrame, assays: dict[str, pd.DataFrame],
    feature_missing: pd.DataFrame, sample_missing: pd.DataFrame,
    threshold: float = 0.20
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], list[str], list[str]]:
    """Remove high-missingness protein features and samples across assays.

    Parameters
    ----------
    cohort : pandas.DataFrame
        Aligned earliest-draw patient metadata.
    assays : dict[str, pandas.DataFrame]
        Assays aligned to ``cohort``.
    feature_missing : pandas.DataFrame
        Per-feature missingness table.
    sample_missing : pandas.DataFrame
        Per-sample missingness table.
    threshold : float, default=0.20
        Exclusive missingness threshold.

    Returns
    -------
    tuple
        Filtered cohort, filtered assays, removed proteins, and removed samples.
    """
    proteins = feature_missing.loc[
        feature_missing["modality"].eq("Protein") &
        feature_missing["missing_fraction"].gt(threshold), "feature"
    ].tolist()
    samples = sample_missing.loc[
        sample_missing["modality"].eq("Protein") &
        sample_missing["missing_fraction"].gt(threshold), "sample_id"
    ].tolist()
    keep = ~cohort["sample_id"].isin(samples)
    filtered_cohort = cohort.loc[keep].reset_index(drop=True)
    filtered = {name: assay.loc[keep].reset_index(drop=True).copy()
                for name, assay in assays.items()}
    filtered["protein"] = filtered["protein"].drop(columns=proteins)
    return filtered_cohort, filtered, proteins, samples


def pca_scores(
    assay: pd.DataFrame, cohort: pd.DataFrame, modality: str
) -> tuple[pd.DataFrame, np.ndarray]:
    """Impute, transform, and compute the first two exploratory PCs.

    Parameters
    ----------
    assay : pandas.DataFrame
        Assay aligned to ``cohort``.
    cohort : pandas.DataFrame
        Corresponding sample metadata.
    modality : str
        Assay modality; protein uses log2(x + 1) for exploratory PCA.

    Returns
    -------
    tuple[pandas.DataFrame, numpy.ndarray]
        Sample scores and percent variance explained by PC1/PC2.
    """
    measured = assay["sample_id"].notna().to_numpy()
    features = assay_feature_columns(assay)
    x = assay.loc[measured, features].copy()
    for feature in features:
        observed = x[feature].dropna()
        if observed.empty:
            continue
        fill = observed.min() / 2 if modality == "protein" else observed.median()
        x[feature] = x[feature].fillna(fill)
    if modality == "protein":
        x = np.log2(x + 1)
    usable = x.notna().all() & x.std(ddof=1).gt(0)
    standardized = (x.loc[:, usable] - x.loc[:, usable].mean()) / x.loc[:, usable].std(ddof=1)
    fit = PCA(n_components=2).fit(standardized)
    scores = cohort.loc[measured, ["sample_id", "stage_group", "site", "plate_id"]].reset_index(drop=True)
    scores["PC1"], scores["PC2"] = fit.transform(standardized).T
    return scores, fit.explained_variance_ratio_ * 100


def plot_pca(scores: pd.DataFrame, variance_pct: np.ndarray,
             modality: str, shape: str = "site") -> plt.Figure:
    """Plot the first two PCs by stage and site or plate.

    Parameters
    ----------
    scores : pandas.DataFrame
        PCA sample scores and metadata.
    variance_pct : numpy.ndarray
        Variance explained by PC1 and PC2.
    modality : str
        Name used in the plot title.
    shape : str, default="site"
        Metadata column used for point styles.

    Returns
    -------
    matplotlib.figure.Figure
        Scatterplot figure.
    """
    fig, ax = plt.subplots(figsize=(6, 4.5))
    sns.scatterplot(data=scores, x="PC1", y="PC2", hue="stage_group", style=shape,
                    alpha=0.8, ax=ax)
    ax.set(xlabel=f"PC1 ({variance_pct[0]:.1f}% variance)",
           ylabel=f"PC2 ({variance_pct[1]:.1f}% variance)",
           title=f"{modality.title()} PCA by {shape}")
    ax.legend(loc="best", fontsize=7)
    fig.tight_layout()
    return fig

Assay coverage in earliest-draw cohort

In [ ]:
assay_coverage, feature_missing, sample_missing = summarize_missingness(
    assays_primary, len(primary)
)
for name, table in (("assay_coverage", assay_coverage),
                    ("feature_missing", feature_missing),
                    ("sample_missing", sample_missing)):
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
display(assay_coverage)

Summaries of per-feature and per-sample missingness are as below

In [ ]:
feature_summary = feature_missing.groupby("modality").agg(
    n_features=("feature", "size"),
    median_missing_fraction=("missing_fraction", "median"),
    features_above_20pct=("missing_fraction", lambda x: (x > 0.2).sum())
).reset_index()
sample_summary = sample_missing.groupby("modality").agg(
    n_samples=("sample_id", "size"),
    median_missing_fraction=("missing_fraction", "median"),
    samples_above_20pct=("missing_fraction", lambda x: (x > 0.2).sum())
).reset_index()
display(feature_summary, sample_summary)

And the distribution plots are shown below, from which we can see only a few features and one sample exceeding the filtering cutoff 0.2.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for modality, subset in feature_missing.groupby("modality"):
    axes[0].hist(subset["missing_fraction"], bins=20, alpha=0.55, label=modality)
for modality, subset in sample_missing.groupby("modality"):
    axes[1].hist(subset["missing_fraction"], bins=20, alpha=0.55, label=modality)
for ax, title in zip(axes, ("Feature missingness", "Sample missingness")):
    ax.axvline(0.2, color="firebrick", linestyle="--")
    ax.set(xlabel="Missing fraction", ylabel="Count", title=title)
    ax.legend()
fig.tight_layout()
plt.show()

primary_filtered, assays_filtered, protein_ids_to_remove, samples_to_remove = filter_qc(
    primary, assays_primary, feature_missing, sample_missing
)
print("Removed proteins:", protein_ids_to_remove)
print("Removed samples:", samples_to_remove)

Methylation regions have at most 5% missingness, and fragmentomics has
none. Three proteins and one sample exceed 20% missingness and are excluded
from subsequent analyses.

### PCA Plots

We generate PCA plots for each modality using the first two components. Points are colored by cancer stage and shaped by site. An additional protein plot uses point shapes to identify plates.

For these exploratory plots, missing methylation and fragmentomics values are filled with feature medians. Missing protein values are filled with half the smallest observed concentration for that protein, then concentrations are log-transformed. PCA uses centered and scaled features with nonzero variance.

In [ ]:
for modality, assay in assays_filtered.items():
    scores, variance_pct = pca_scores(assay, primary_filtered, modality)
    plot_pca(scores, variance_pct, modality)
    if modality == "protein":
        plot_pca(scores, variance_pct, modality, shape="plate_id")
plt.show()

The plots suggest that:

- There are no obvious sample outliers.
- In the methylation data, site A separates from sites B and C along the first component regardless of cancer status, suggesting a potential batch effect.
- In the protein data, PLT02 samples (triangles) tend to cluster near the top regardless of cancer status, also suggesting a potential batch effect.

### Detect features with batch effects

We next assess which methylation regions vary by site and which proteins vary
by plate. Using control samples only, we fit a linear model for each feature
with site, age, and sex as predictors; the protein models also include plate.
We then identify features associated with batch variables after accounting
for age and sex. For
those features, we subtract the fitted batch effects from the original values
and save the original and adjusted values alongside the fitted coefficients.


In [ ]:
def make_batch_design(
    metadata: pd.DataFrame, site_levels: list[str],
    plate_levels: list[str] | None = None
) -> pd.DataFrame:
    """Build a fixed-reference design for age, sex, site, and optional plate.

    Parameters
    ----------
    metadata : pandas.DataFrame
        Patient age, sex, site, and optionally plate ID.
    site_levels : list[str]
        Training-site levels in reference-first order.
    plate_levels : list[str] or None, default=None
        Training-plate levels for proteins.

    Returns
    -------
    pandas.DataFrame
        Numeric design with the same columns for train and test patients.
    """
    if not metadata["site"].isin(site_levels).all():
        raise ValueError("Unseen validation site.")
    columns: dict[str, Any] = {"(Intercept)": np.ones(len(metadata)),
                               "age": metadata["age"].to_numpy(dtype=float),
                               "sexM": metadata["sex"].eq("M").to_numpy(dtype=float)}
    for level in site_levels[1:]:
        columns[f"site{level}"] = metadata["site"].eq(level).to_numpy(dtype=float)
    if plate_levels is not None:
        if not metadata["plate_id"].isin(plate_levels).all():
            raise ValueError("Unseen validation plate.")
        for level in plate_levels[1:]:
            columns[f"plate_id{level}"] = metadata["plate_id"].eq(level).to_numpy(dtype=float)
    return pd.DataFrame(columns, index=metadata.index)


def bh_adjust(p_values: np.ndarray) -> np.ndarray:
    """Adjust finite p-values with the Benjamini-Hochberg procedure.

    Parameters
    ----------
    p_values : numpy.ndarray
        Raw p-values, optionally containing NaNs.

    Returns
    -------
    numpy.ndarray
        Adjusted p-values in input order.
    """
    values = np.asarray(p_values, dtype=float)
    result = np.full(values.shape, np.nan)
    finite = np.flatnonzero(np.isfinite(values))
    if len(finite):
        order = finite[np.argsort(values[finite])]
        scaled = values[order] * len(order) / np.arange(1, len(order) + 1)
        result[order] = np.minimum(1, np.minimum.accumulate(scaled[::-1])[::-1])
    return result


def fit_control_batch(
    assay: pd.DataFrame, cohort: pd.DataFrame, modality: str
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    """Test site/plate effects feature by feature in observed controls.

    Parameters
    ----------
    assay : pandas.DataFrame
        Assay aligned to ``cohort``.
    cohort : pandas.DataFrame
        Sample metadata including age, sex, site, stage, and plate.
    modality : str
        ``methylation`` or ``protein``.

    Returns
    -------
    tuple
        Batch-test table, coefficient table, and fitted model metadata.
    """
    measured = assay["sample_id"].notna().to_numpy()
    controls = cohort["stage_group"].eq("Control").to_numpy() & measured
    site_levels = sorted(cohort.loc[measured, "site"].unique().tolist())
    plate_levels = (sorted(cohort.loc[measured, "plate_id"].unique().tolist())
                    if modality == "protein" else None)
    variables = ["site", "plate_id"] if modality == "protein" else ["site"]
    tests, coefficients, fits = [], [], {}
    for feature in assay_feature_columns(assay):
        original = assay[feature].to_numpy(dtype=float)
        use = controls & np.isfinite(original)
        if modality == "protein":
            use &= original > 0
        y = np.log2(original[use]) if modality == "protein" else original[use]
        design = make_batch_design(cohort.loc[use], site_levels, plate_levels)
        n = len(y)
        pvals = {name: np.nan for name in variables}
        if n > design.shape[1] and np.linalg.matrix_rank(design.to_numpy()) == design.shape[1]:
            beta = np.linalg.lstsq(design.to_numpy(), y, rcond=None)[0]
            residual_ss = float(np.sum((y - design.to_numpy() @ beta) ** 2))
            residual_df = n - design.shape[1]
            fits[feature] = dict(beta=beta, columns=design.columns.tolist())
            coefficients.extend(dict(modality=modality, feature=feature,
                                     model_scale="log2" if modality == "protein" else "original",
                                     term=term, estimate=float(value))
                                for term, value in zip(design.columns, beta))
            for variable in variables:
                removed = [c for c in design if c.startswith(variable)]
                reduced = design.drop(columns=removed).to_numpy()
                reduced_beta = np.linalg.lstsq(reduced, y, rcond=None)[0]
                reduced_ss = float(np.sum((y - reduced @ reduced_beta) ** 2))
                df1 = len(removed)
                if df1:
                    statistic = max(0.0, reduced_ss - residual_ss) / df1
                    statistic /= max(residual_ss / residual_df, np.finfo(float).eps)
                    pvals[variable] = float(f_distribution.sf(statistic, df1, residual_df))
        tests.extend(dict(modality=modality, feature=feature, batch_variable=name,
                          n_controls=n, p_value=pvals[name]) for name in variables)
    test_table = pd.DataFrame(tests)
    test_table["q_value"] = np.nan
    for variable in variables:
        mask = test_table["batch_variable"].eq(variable)
        test_table.loc[mask, "q_value"] = bh_adjust(test_table.loc[mask, "p_value"].to_numpy())
    test_table["significant"] = test_table["q_value"].lt(0.05)
    fit_info = dict(fits=fits, site_levels=site_levels, plate_levels=plate_levels,
                    modality=modality)
    return test_table, pd.DataFrame(coefficients), fit_info


def apply_control_batch(
    assay: pd.DataFrame, cohort: pd.DataFrame, test_table: pd.DataFrame,
    fit_info: dict[str, Any]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Subtract significant control-derived batch terms from assay values.

    Parameters
    ----------
    assay : pandas.DataFrame
        Input assay aligned to ``cohort``.
    cohort : pandas.DataFrame
        Patient metadata.
    test_table : pandas.DataFrame
        Feature-level significance decisions.
    fit_info : dict[str, Any]
        Output of :func:`fit_control_batch`.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame]
        Adjusted assay and original/adjusted values for affected features.
    """
    adjusted = assay.copy()
    measured = assay["sample_id"].notna().to_numpy()
    design = make_batch_design(cohort.loc[measured], fit_info["site_levels"],
                               fit_info["plate_levels"])
    values = []
    for feature, rows in test_table[test_table["significant"]].groupby("feature"):
        fit = fit_info["fits"][feature]
        terms = [term for term in fit["columns"]
                 if any(term.startswith(v) for v in rows["batch_variable"])]
        term_indices = [fit["columns"].index(term) for term in terms]
        effect = design[terms].to_numpy() @ fit["beta"][term_indices]
        original = assay.loc[measured, feature].to_numpy(dtype=float)
        if fit_info["modality"] == "protein":
            new_value = np.exp2(np.log2(original) - effect)
        else:
            new_value = original - effect
        adjusted.loc[measured, feature] = new_value
        for i, sample_id in enumerate(cohort.loc[measured, "sample_id"]):
            row = dict(modality=fit_info["modality"], feature=feature,
                       sample_id=sample_id, site=cohort.loc[measured, "site"].iloc[i],
                       plate_id=(cohort.loc[measured, "plate_id"].iloc[i]
                                 if fit_info["modality"] == "protein" else np.nan),
                       original_value=original[i], adjusted_value=new_value[i],
                       estimated_batch_effect=effect[i],
                       effect_scale="log2" if fit_info["modality"] == "protein" else "original",
                       adjusted_for="+".join(rows["batch_variable"]))
            row.update({"coef_" + term: value for term, value in
                        zip(fit["columns"], fit["beta"])})
            values.append(row)
    return adjusted, pd.DataFrame(values)

In [ ]:
batch_tests, batch_coefficients = [], []
assays_batch_adjusted = assays_filtered.copy()
batch_value_rows = []
for modality in ("methylation", "protein"):
    tests, coefficients, fit_info = fit_control_batch(
        assays_filtered[modality], primary_filtered, modality
    )
    adjusted, values = apply_control_batch(
        assays_filtered[modality], primary_filtered, tests, fit_info
    )
    batch_tests.append(tests)
    batch_coefficients.append(coefficients)
    batch_value_rows.append(values)
    assays_batch_adjusted[modality] = adjusted
batch_effect_tests = pd.concat(batch_tests, ignore_index=True)
batch_effect_coefficients = pd.concat(batch_coefficients, ignore_index=True)
batch_adjusted_values = pd.concat(batch_value_rows, ignore_index=True)
for name, table in (("batch_effect_tests", batch_effect_tests),
                    ("batch_effect_coefficients", batch_effect_coefficients),
                    ("batch_adjusted_values", batch_adjusted_values)):
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
display(batch_effect_tests.groupby(["modality", "batch_variable"]).agg(
    n_tested=("p_value", "count"), n_significant=("significant", "sum")
).reset_index())

These tests identify 39 methylation regions associated with site and two proteins associated with plate after accounting for the other factors.

We then repeat PCA using the adjusted values:

In [ ]:
for modality, assay in assays_batch_adjusted.items():
    scores, variance_pct = pca_scores(assay, primary_filtered, modality)
    plot_pca(scores, variance_pct, modality)
    if modality == "protein":
        plot_pca(scores, variance_pct, modality, shape="plate_id")
plt.show()

After adjustment, the PCA plots suggest that:

    The protein plate effect appears to have been removed.
    Methylation samples still separate by site, but less strongly, as indicated by the variance explained by PC1. Cancer status appears to contribute more to the remaining separation.

To assess the remaining site separation, we plot PCA of the adjusted methylation data separately for controls and cancer samples, coloring each point by site.

In [ ]:
for stage_label, mask in (("Controls", primary_filtered["stage_group"].eq("Control")),
                          ("Cancer", primary_filtered["stage_group"].ne("Control"))):
    scores, variance_pct = pca_scores(
        assays_batch_adjusted["methylation"].loc[mask].reset_index(drop=True),
        primary_filtered.loc[mask].reset_index(drop=True), "methylation"
    )
    plot_pca(scores, variance_pct, f"Adjusted methylation: {stage_label}")
plt.show()

In these plots, controls from different sites are closer together, while cancer samples still show some site separation. This may reflect biological differences between sites or incomplete batch correction: only methylation features with FDR < 0.05 were adjusted, and other features may still carry batch effects.

### Repeated nested cross-validation with LASSO

Because protein plate data are available only for sites A and B, feature selection and model training use samples from those sites. LASSO selects features while fitting the model, which can favor markers that contribute complementary predictive information. All features are standardized so that the penalty is applied on a comparable scale. Repeated nested cross-validation helps narrow the panel to at most five methylation and three protein markers.

Sample splits are stratified by cancer stage, site, and plate where possible.

To address site and plate effects during modeling, we fit batch-correction models using controls within each training fold. We then use those fitted models to adjust all samples in that fold.

For validation, every imputation value, control-derived batch coefficient, and standardization parameter is fitted again within its training fold. After LASSO selection, each fold keeps the five largest nonzero methylation coefficients and three largest nonzero protein coefficients. Held-out predictions use only those retained coefficients and the fitted intercept. This cap is part of the validated prediction procedure; the full-cohort adjusted values above are used for exploratory QC only.

In [ ]:
@dataclass
class PreprocessBlock:
    """Training-derived transformation for one assay modality.

    Parameters
    ----------
    features : list[str]
        Retained input features.
    center : numpy.ndarray
        Training means after imputation and batch correction.
    spread : numpy.ndarray
        Training standard deviations.
    fill : numpy.ndarray or None
        Training-only imputation values.
    site_levels : list[str] or None
        Known sites for the batch design.
    plate_levels : list[str] or None
        Known plates for the batch design.
    batch_coefficients : pandas.DataFrame or None
        Control-derived coefficients for significant batch terms.
    """
    features: list[str]
    center: np.ndarray
    spread: np.ndarray
    fill: np.ndarray | None = None
    site_levels: list[str] | None = None
    plate_levels: list[str] | None = None
    batch_coefficients: pd.DataFrame | None = None


@dataclass
class LassoPanel:
    """Sparse logistic panel after modality-specific feature caps.

    Parameters
    ----------
    intercept : float
        Intercept from the high-dimensional LASSO fit.
    coefficients : pandas.Series
        Retained coefficients; excluded coefficients are set to zero.
    fraction : float
        Penalty divided by the full-training ``lambda_max``.
    """
    intercept: float
    coefficients: pd.Series
    fraction: float


def prepare_model_inputs(
    cohort: pd.DataFrame, assays: dict[str, pd.DataFrame]
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """Select site A/B patients with protein rows and align raw model matrices.

    Parameters
    ----------
    cohort : pandas.DataFrame
        QC-filtered earliest-draw metadata.
    assays : dict[str, pandas.DataFrame]
        QC-filtered, aligned assays.

    Returns
    -------
    tuple[pandas.DataFrame, dict[str, pandas.DataFrame]]
        Modeling metadata and raw methylation/protein predictors.
    """
    keep = cohort["site"].isin(["A", "B"]) & assays["protein"]["sample_id"].notna()
    model_cohort = cohort.loc[keep].reset_index(drop=True).copy()
    raw = {modality: assays[modality].loc[keep, assay_feature_columns(assays[modality])]
           .reset_index(drop=True).astype(float)
           for modality in ("methylation", "protein")}
    if model_cohort["patient_id"].duplicated().any() or model_cohort["y"].nunique() != 2:
        raise ValueError("Invalid modeling cohort.")
    return model_cohort, raw


def make_stratified_folds(
    cohort: pd.DataFrame, k: int, seed: int, trials: int = 50
) -> np.ndarray:
    """Balance stage/site/plate strata and class counts across CV folds.

    Parameters
    ----------
    cohort : pandas.DataFrame
        Patient-level modeling metadata.
    k : int
        Number of folds.
    seed : int
        Random seed for reproducible assignments.
    trials : int, default=50
        Candidate assignments compared for marginal balance.

    Returns
    -------
    numpy.ndarray
        Zero-based fold ID for each patient.
    """
    n = len(cohort)
    if k < 2 or n < k or cohort["patient_id"].duplicated().any():
        raise ValueError("Invalid fold request or duplicate patient IDs.")
    rng = np.random.default_rng(seed)
    strata = cohort[["stage_group", "site", "plate_id"]].astype(str).agg("|".join, axis=1)
    groups = [np.flatnonzero(strata.eq(value)) for value in strata.unique()]
    y = cohort["y"].to_numpy()
    best, best_score = None, np.inf
    for _ in range(trials):
        fold_ids = np.empty(n, dtype=int)
        for group in groups:
            shuffled = rng.permutation(group)
            fold_ids[shuffled] = np.resize(rng.permutation(k), len(group))
        if any(len(np.unique(y[fold_ids == fold])) < 2 for fold in range(k)):
            continue
        score = np.sum((np.bincount(fold_ids, minlength=k) - n / k) ** 2)
        for variable in ("stage_group", "site", "plate_id"):
            for level in cohort[variable].astype(str).unique():
                counts = np.bincount(fold_ids[cohort[variable].astype(str).eq(level)],
                                     minlength=k)
                score += np.sum((counts - np.sum(counts) / k) ** 2)
        if score < best_score:
            best, best_score = fold_ids.copy(), score
    if best is None:
        raise ValueError("Could not construct folds containing both classes.")
    return best


def fit_model_preprocess(
    cohort: pd.DataFrame, raw_data: dict[str, pd.DataFrame], train_idx: np.ndarray
) -> dict[str, PreprocessBlock]:
    """Learn fold-specific imputation, control batch effects, and scaling.

    Parameters
    ----------
    cohort : pandas.DataFrame
        Complete modeling metadata.
    raw_data : dict[str, pandas.DataFrame]
        Aligned raw predictors keyed by modality.
    train_idx : numpy.ndarray
        Positional indices for training patients only.

    Returns
    -------
    dict[str, PreprocessBlock]
        Training-derived transformation for each modality.
    """
    training = cohort.iloc[train_idx].reset_index(drop=True)
    controls = training["y"].eq(0).to_numpy()
    if controls.sum() < 10:
        raise ValueError("Too few training controls.")
    blocks = {}
    for modality, frame in raw_data.items():
        x_frame = frame.iloc[train_idx].reset_index(drop=True)
        if modality == "fragmentomics":
            x = x_frame.to_numpy(dtype=float)
            if not np.isfinite(x).all():
                raise ValueError("Fragmentomics inputs must be complete and finite.")
            center = x.mean(axis=0)
            spread = x.std(axis=0, ddof=1)
            usable = np.isfinite(spread) & (spread > 1e-8)
            if not usable.any():
                raise ValueError("No variable fragmentomics features.")
            blocks[modality] = PreprocessBlock(x_frame.columns[usable].tolist(),
                                               center[usable], spread[usable])
            continue
        original = x_frame.to_numpy(dtype=float, copy=True)
        fills = np.empty(original.shape[1])
        for j in range(original.shape[1]):
            observed = original[np.isfinite(original[:, j]), j]
            if modality == "protein":
                observed = observed[observed > 0]
            fills[j] = (np.min(observed) / 2 if modality == "protein" else np.median(observed)) \
                       if len(observed) else np.nan
        retain = np.isfinite(fills)
        if not retain.any():
            raise ValueError(f"No usable {modality} features.")
        features = x_frame.columns[retain].tolist()
        x = original[:, retain]
        fill = fills[retain]
        missing = np.isnan(x)
        x[missing] = np.broadcast_to(fill, x.shape)[missing]
        if modality == "protein":
            if np.any(x <= 0):
                raise ValueError("Protein concentrations must be positive before log2.")
            x = np.log2(x)
        site_levels = sorted(training["site"].unique().tolist())
        plate_levels = (sorted(training["plate_id"].unique().tolist())
                        if modality == "protein" else None)
        design = make_batch_design(training, site_levels, plate_levels)
        control_design = design.loc[controls].to_numpy()
        if np.linalg.matrix_rank(control_design) < control_design.shape[1]:
            raise ValueError("Control batch design is rank deficient in this fold.")
        coefficients = np.linalg.lstsq(control_design, x[controls], rcond=None)[0]
        residual = x[controls] - control_design @ coefficients
        residual_ss = np.sum(residual ** 2, axis=0)
        residual_df = controls.sum() - control_design.shape[1]
        corrected = coefficients.copy()
        variables = ("site", "plate_id") if modality == "protein" else ("site",)
        for variable in variables:
            term_rows = [i for i, name in enumerate(design.columns) if name.startswith(variable)]
            reduced = np.delete(control_design, term_rows, axis=1)
            reduced_coef = np.linalg.lstsq(reduced, x[controls], rcond=None)[0]
            reduced_ss = np.sum((x[controls] - reduced @ reduced_coef) ** 2, axis=0)
            statistic = np.maximum(0, reduced_ss - residual_ss) / len(term_rows)
            statistic /= np.maximum(residual_ss / residual_df, np.finfo(float).eps)
            q_value = bh_adjust(f_distribution.sf(statistic, len(term_rows), residual_df))
            corrected[np.ix_(term_rows, np.flatnonzero(~(q_value < 0.05)))] = 0
        batch_rows = [i for i, name in enumerate(design.columns)
                      if name.startswith("site") or name.startswith("plate_id")]
        adjusted = x - design.to_numpy()[:, batch_rows] @ corrected[batch_rows]
        center = adjusted.mean(axis=0)
        spread = adjusted.std(axis=0, ddof=1)
        usable = np.isfinite(spread) & (spread > 1e-8)
        if not usable.any():
            raise ValueError(f"No variable {modality} features.")
        blocks[modality] = PreprocessBlock(
            np.asarray(features)[usable].tolist(), center[usable], spread[usable],
            fill[usable], site_levels, plate_levels,
            pd.DataFrame(corrected[:, usable], index=design.columns,
                         columns=np.asarray(features)[usable])
        )
    return blocks


def apply_model_preprocess(
    cohort: pd.DataFrame, raw_data: dict[str, pd.DataFrame],
    blocks: dict[str, PreprocessBlock], idx: np.ndarray
) -> pd.DataFrame:
    """Apply training-derived transformations to any patient subset.

    Parameters
    ----------
    cohort : pandas.DataFrame
        Complete modeling metadata.
    raw_data : dict[str, pandas.DataFrame]
        Aligned raw predictors.
    blocks : dict[str, PreprocessBlock]
        Transformations fitted on a training fold.
    idx : numpy.ndarray
        Positional indices to transform.

    Returns
    -------
    pandas.DataFrame
        Standardized predictors with modality-prefixed names.
    """
    metadata = cohort.iloc[idx].reset_index(drop=True)
    transformed = []
    for modality, block in blocks.items():
        # Imputation writes into x; own the array even with pandas copy-on-write.
        x = raw_data[modality].iloc[idx][block.features].to_numpy(dtype=float, copy=True)
        if modality == "fragmentomics":
            if not np.isfinite(x).all():
                raise ValueError("Fragmentomics inputs must be complete and finite.")
            adjusted = x
        else:
            missing = np.isnan(x)
            x[missing] = np.broadcast_to(block.fill, x.shape)[missing]
            if modality == "protein":
                x = np.log2(x)
            design = make_batch_design(metadata, block.site_levels,
                                       block.plate_levels)
            coefficients = block.batch_coefficients
            batch_terms = [name for name in coefficients.index
                           if name.startswith("site") or name.startswith("plate_id")]
            adjusted = x - design[batch_terms].to_numpy() @ coefficients.loc[batch_terms].to_numpy()
        scaled = (adjusted - block.center) / block.spread
        transformed.append(pd.DataFrame(scaled,
                         columns=[f"{modality}__{feature}" for feature in block.features]))
    return pd.concat(transformed, axis=1)


def binary_log_loss(y: np.ndarray, probability: np.ndarray) -> float:
    """Compute mean binary cross-entropy with bounded probabilities.

    Parameters
    ----------
    y : numpy.ndarray
        Binary outcomes.
    probability : numpy.ndarray
        Predicted cancer probabilities.

    Returns
    -------
    float
        Mean binary log loss.
    """
    probability = np.clip(probability, 1e-6, 1 - 1e-6)
    return float(np.mean(-y * np.log(probability) - (1 - y) * np.log1p(-probability)))


def lambda_max_binomial(x: np.ndarray, y: np.ndarray) -> float:
    """Find the L1 penalty that first sets all logistic slopes to zero.

    Parameters
    ----------
    x : numpy.ndarray
        Standardized training predictors.
    y : numpy.ndarray
        Binary training outcomes.

    Returns
    -------
    float
        Approximate binomial ``lambda_max`` for the mean-loss objective.
    """
    value = float(np.max(np.abs(x.T @ (y - y.mean()))) / len(y))
    if not np.isfinite(value) or value <= 0:
        raise ValueError("Cannot derive a positive lambda_max.")
    return value


def fit_lasso_fraction(
    x: np.ndarray, y: np.ndarray, fraction: float, lambda_max: float,
    seed: int, previous: LogisticRegression | None = None
) -> LogisticRegression:
    """Fit L1 logistic regression at a fraction of training lambda_max.

    Parameters
    ----------
    x : numpy.ndarray
        Training predictors already standardized within the fold.
    y : numpy.ndarray
        Binary outcomes.
    fraction : float
        Penalty fraction of ``lambda_max``.
    lambda_max : float
        Training-derived all-zero threshold.
    seed : int
        Solver seed.
    previous : sklearn.linear_model.LogisticRegression or None, default=None
        Previous stronger-penalty fit, used for warm starts.

    Returns
    -------
    sklearn.linear_model.LogisticRegression
        Fitted sparse logistic model.
    """
    c_value = 1.0 / (len(y) * lambda_max * fraction)
    if previous is None:
        # scikit-learn 1.8+ deprecates ``penalty`` in favor of ``l1_ratio``.
        from sklearn import __version__ as sklearn_version
        version = tuple(int(part) for part in sklearn_version.split(".")[:2])
        penalty_args = ({"l1_ratio": 1.0} if version >= (1, 8)
                        else {"penalty": "l1"})
        model = LogisticRegression(
            solver="saga", fit_intercept=True, warm_start=True,
            max_iter=2500, tol=1e-3, random_state=seed, **penalty_args
        )
    else:
        model = previous
    model.set_params(C=c_value)
    model.fit(x, y)
    return model


def cv_lasso(
    cohort: pd.DataFrame, raw_data: dict[str, pd.DataFrame],
    train_idx: np.ndarray, fold_ids: np.ndarray, fractions: tuple[float, ...],
    seed: int
) -> tuple[float, float, pd.DataFrame]:
    """Choose a penalty fraction with fold-contained preprocessing and one SE.

    Parameters
    ----------
    cohort : pandas.DataFrame
        Patient metadata.
    raw_data : dict[str, pandas.DataFrame]
        Aligned raw predictors.
    train_idx : numpy.ndarray
        Patients available to inner CV.
    fold_ids : numpy.ndarray
        Inner fold assignments aligned to ``train_idx``.
    fractions : tuple[float, ...]
        Descending fractions of each inner training fold's ``lambda_max``.
    seed : int
        Solver seed.

    Returns
    -------
    tuple[float, float, pandas.DataFrame]
        Selected fraction, its weighted mean log loss, and tuning curve.
    """
    if len(train_idx) != len(fold_ids):
        raise ValueError("Fold IDs and training indices have different lengths.")
    folds = np.unique(fold_ids)
    fold_loss = np.empty((len(folds), len(fractions)))
    fold_size = np.empty(len(folds), dtype=int)
    for i, fold in enumerate(folds):
        inner_train = train_idx[fold_ids != fold]
        inner_valid = train_idx[fold_ids == fold]
        prep = fit_model_preprocess(cohort, raw_data, inner_train)
        x_train = apply_model_preprocess(cohort, raw_data, prep, inner_train).to_numpy()
        x_valid = apply_model_preprocess(cohort, raw_data, prep, inner_valid).to_numpy()
        y_train = cohort.iloc[inner_train]["y"].to_numpy()
        y_valid = cohort.iloc[inner_valid]["y"].to_numpy()
        max_penalty = lambda_max_binomial(x_train, y_train)
        fit = None
        for j, fraction in enumerate(fractions):
            fit = fit_lasso_fraction(x_train, y_train, fraction, max_penalty,
                                     seed + i, fit)
            probability = fit.predict_proba(x_valid)[:, 1]
            fold_loss[i, j] = binary_log_loss(y_valid, probability)
        fold_size[i] = len(inner_valid)
    mean_loss = np.average(fold_loss, axis=0, weights=fold_size)
    best = int(np.argmin(mean_loss))
    cutoff = mean_loss[best] + np.std(fold_loss[:, best], ddof=1) / np.sqrt(len(folds))
    chosen = int(np.flatnonzero(mean_loss <= cutoff)[0])
    curve = pd.DataFrame({"lambda_fraction": fractions, "mean_logloss": mean_loss})
    return float(fractions[chosen]), float(mean_loss[chosen]), curve


def select_lasso_panel(
    cohort: pd.DataFrame, raw_data: dict[str, pd.DataFrame],
    train_idx: np.ndarray, prep: dict[str, PreprocessBlock],
    tuned_fraction: float, fractions: tuple[float, ...], seed: int,
    caps: dict[str, int]
) -> LassoPanel:
    """Keep the largest nonzero LASSO slopes within each modality cap.

    Parameters
    ----------
    cohort : pandas.DataFrame
        Patient metadata.
    raw_data : dict[str, pandas.DataFrame]
        Aligned raw predictors.
    train_idx : numpy.ndarray
        Full outer-training or full-cohort indices.
    prep : dict[str, PreprocessBlock]
        Training-only preprocessing rules.
    tuned_fraction : float
        One-SE penalty fraction chosen by inner CV.
    fractions : tuple[float, ...]
        Descending candidate penalty fractions.
    seed : int
        Solver seed.
    caps : dict[str, int]
        Maximum selected features for each modality.

    Returns
    -------
    LassoPanel
        Intercept and capped coefficients from the fitted LASSO model.
    """
    x_frame = apply_model_preprocess(cohort, raw_data, prep, train_idx)
    x = x_frame.to_numpy()
    y = cohort.iloc[train_idx]["y"].to_numpy()
    max_penalty = lambda_max_binomial(x, y)
    fit = None
    for fraction in (f for f in fractions if f <= tuned_fraction + 1e-12):
        fit = fit_lasso_fraction(x, y, fraction, max_penalty, seed, fit)
        slopes = pd.Series(fit.coef_.ravel(), index=x_frame.columns)
        ranked = slopes.loc[slopes.abs() > 1e-6].abs().sort_values(ascending=False).index
        selected = []
        for modality, cap in caps.items():
            selected.extend([name for name in ranked if name.startswith(modality + "__")][:cap])
        if selected:
            return LassoPanel(float(fit.intercept_[0]), slopes.loc[selected], fraction)
    raise ValueError("LASSO selected no markers in this training fold.")


def predict_lasso_panel(panel: LassoPanel, x: pd.DataFrame) -> np.ndarray:
    """Predict cancer probability using only the capped panel coefficients.

    Parameters
    ----------
    panel : LassoPanel
        Fitted capped panel.
    x : pandas.DataFrame
        Preprocessed predictors for test patients.

    Returns
    -------
    numpy.ndarray
        Cancer probabilities.
    """
    return expit(panel.intercept + x[panel.coefficients.index].to_numpy() @
                 panel.coefficients.to_numpy())


def run_nested_lasso(
    cohort: pd.DataFrame, raw_data: dict[str, pd.DataFrame],
    repeats: int, outer_folds: int, inner_folds: int, seed: int,
    fractions: tuple[float, ...], caps: dict[str, int]
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run repeated nested CV with training-only preprocessing and selection.

    Parameters
    ----------
    cohort : pandas.DataFrame
        Patient-level modeling metadata.
    raw_data : dict[str, pandas.DataFrame]
        Aligned raw predictors.
    repeats : int
        Number of outer CV repetitions.
    outer_folds : int
        Held-out folds per repetition.
    inner_folds : int
        Tuning folds within each outer training set.
    seed : int
        Base random seed.
    fractions : tuple[float, ...]
        Descending candidate lambda fractions.
    caps : dict[str, int]
        Marker cap for each modality.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame, pandas.DataFrame]
        Held-out predictions, outer selections, and tuning records.
    """
    predictions, selections, tuning = [], [], []
    for repeat_id in range(1, repeats + 1):
        outer_id = make_stratified_folds(cohort, outer_folds, seed + repeat_id)
        for fold in range(outer_folds):
            train_idx = np.flatnonzero(outer_id != fold)
            test_idx = np.flatnonzero(outer_id == fold)
            inner_id = make_stratified_folds(cohort.iloc[train_idx].reset_index(drop=True),
                                              inner_folds, seed + 1000 * repeat_id + fold + 1)
            fraction, inner_loss, _ = cv_lasso(
                cohort, raw_data, train_idx, inner_id, fractions, seed + repeat_id + fold
            )
            prep = fit_model_preprocess(cohort, raw_data, train_idx)
            panel = select_lasso_panel(cohort, raw_data, train_idx, prep, fraction,
                                       fractions, seed + repeat_id + fold, caps)
            x_test = apply_model_preprocess(cohort, raw_data, prep, test_idx)
            probability = predict_lasso_panel(panel, x_test)
            for j, idx in enumerate(test_idx):
                row = cohort.iloc[idx]
                predictions.append(dict(repeat_id=repeat_id, fold=fold + 1,
                    patient_id=row["patient_id"], sample_id=row["sample_id"],
                    stage=str(row["stage_group"]), site=row["site"],
                    plate_id=row["plate_id"], cancer_status=int(row["y"]),
                    probability=float(probability[j])))
            for feature, coefficient in panel.coefficients.items():
                selections.append(dict(repeat_id=repeat_id, fold=fold + 1,
                    feature=feature, modality=feature.split("__", 1)[0],
                    coefficient=float(coefficient)))
            counts = {f"n_{modality}": sum(name.startswith(modality + "__")
                     for name in panel.coefficients.index) for modality in
                     ("methylation", "protein", "fragmentomics")}
            tuning.append(dict(repeat_id=repeat_id, fold=fold + 1,
                inner_folds=inner_folds, n_train=len(train_idx), n_test=len(test_idx),
                inner_logloss=inner_loss, lambda_fraction=panel.fraction, **counts))
    return pd.DataFrame(predictions), pd.DataFrame(selections), pd.DataFrame(tuning)

In [ ]:
def score_repeated_predictions(
    predictions: pd.DataFrame, target_specificity: float = 0.95
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Score pooled and site-specific predictions by repeat and cancer stage.

    Parameters
    ----------
    predictions : pandas.DataFrame
        One held-out probability per patient per repetition.
    target_specificity : float, default=0.95
        Target control specificity for the descriptive cutoff.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame]
        Per-repeat metrics and mean/SD summaries across repetitions.
    """
    required = ["repeat_id", "patient_id", "site", "stage", "cancer_status", "probability"]
    if predictions[required].isna().any().any() or predictions.duplicated(
        ["repeat_id", "patient_id"]
    ).any():
        raise ValueError("Invalid or duplicated held-out predictions.")
    if not predictions["probability"].between(0, 1).all():
        raise ValueError("Predicted probabilities must lie in [0, 1].")
    rows = []
    groups = ["All cancers", "I", "II", "III", "IV"]
    sites = ["All sites"] + sorted(predictions["site"].unique().tolist())
    for repeat_id, repeat_data in predictions.groupby("repeat_id", sort=True):
        for site in sites:
            subset = (repeat_data if site == "All sites" else
                      repeat_data.loc[repeat_data["site"].eq(site)])
            controls = subset.loc[subset["cancer_status"].eq(0)]
            cases = subset.loc[subset["cancer_status"].eq(1)]
            if controls.empty:
                raise ValueError("Each site group must contain controls.")
            ordered = np.sort(controls["probability"].to_numpy())
            cutoff = ordered[int(np.ceil(target_specificity * len(ordered))) - 1]
            specificity = float(np.mean(controls["probability"] <= cutoff))
            for group in groups:
                group_cases = (cases if group == "All cancers" else
                               cases.loc[cases["stage"].eq(group)])
                y = np.r_[np.zeros(len(controls), dtype=int),
                          np.ones(len(group_cases), dtype=int)]
                scores = np.r_[controls["probability"].to_numpy(),
                               group_cases["probability"].to_numpy()]
                positive = scores > cutoff
                tp = int(np.sum((y == 1) & positive))
                fn = int(np.sum((y == 1) & ~positive))
                tn = int(np.sum((y == 0) & ~positive))
                fp = int(np.sum((y == 0) & positive))
                denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
                rows.append(dict(repeat_id=repeat_id, site_group=site,
                    cancer_group=group, n_patients=len(y), n_controls=len(controls),
                    n_cases=len(group_cases), cutoff=cutoff,
                    AUC=roc_auc_score(y, scores) if len(group_cases) else np.nan,
                    MCC=(tp * tn - fp * fn) / denom if denom else np.nan,
                    sensitivity_at_95_specificity=(tp / len(group_cases)
                                                   if len(group_cases) else np.nan),
                    achieved_specificity=specificity))
    per_repeat = pd.DataFrame(rows)
    grouped = per_repeat.groupby(["site_group", "cancer_group"], sort=False)
    summary = grouped.agg(n_repeats=("repeat_id", "nunique"),
                          mean_n_controls=("n_controls", "mean"),
                          mean_n_cases=("n_cases", "mean"))
    for metric in ("AUC", "MCC", "sensitivity_at_95_specificity", "achieved_specificity"):
        summary[f"{metric}_mean"] = grouped[metric].mean()
        summary[f"{metric}_sd"] = grouped[metric].std(ddof=1)
    return per_repeat, summary.reset_index()


def selection_frequency(selections: pd.DataFrame, n_outer_folds: int) -> pd.DataFrame:
    """Count how often each marker was selected across outer training folds.

    Parameters
    ----------
    selections : pandas.DataFrame
        Feature rows from repeated nested CV.
    n_outer_folds : int
        Total number of outer model fits.

    Returns
    -------
    pandas.DataFrame
        Selection counts and frequencies.
    """
    count = (selections.groupby(["modality", "feature"]).size()
             .rename("n_outer_folds").reset_index())
    count["frequency"] = count["n_outer_folds"] / n_outer_folds
    return count.sort_values(["frequency", "modality", "feature"],
                             ascending=[False, True, True]).reset_index(drop=True)


def save_model_results(
    prefix: str, predictions: pd.DataFrame, selections: pd.DataFrame,
    tuning: pd.DataFrame, output_dir: Path
) -> dict[str, pd.DataFrame]:
    """Score and save repeated outer-CV outputs using R-report filenames.

    Parameters
    ----------
    prefix : str
        File prefix such as ``molecular`` or ``fragmentomics``.
    predictions : pandas.DataFrame
        Held-out patient predictions.
    selections : pandas.DataFrame
        Outer-fold selected markers.
    tuning : pandas.DataFrame
        Outer-fold tuning records.
    output_dir : Path
        Directory receiving CSV artifacts.

    Returns
    -------
    dict[str, pandas.DataFrame]
        Scored tables, summaries, and selection frequencies.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions.to_csv(output_dir / f"{prefix}_outer_predictions.csv", index=False)
    selections.to_csv(output_dir / f"{prefix}_outer_selections.csv", index=False)
    tuning.to_csv(output_dir / f"{prefix}_outer_tuning.csv", index=False)
    per_repeat, summary = score_repeated_predictions(predictions)
    pooled = per_repeat.loc[per_repeat["site_group"].eq("All sites") &
                            per_repeat["cancer_group"].eq("All cancers")]
    frequency = selection_frequency(selections, len(tuning))
    pooled.to_csv(output_dir / f"{prefix}_outer_performance.csv", index=False)
    per_repeat.to_csv(output_dir / f"{prefix}_outer_stage_performance.csv", index=False)
    per_repeat.loc[per_repeat["cancer_group"].eq("All cancers")].to_csv(
        output_dir / f"{prefix}_outer_site_performance.csv", index=False)
    summary.to_csv(output_dir / f"{prefix}_outer_performance_summary.csv", index=False)
    summary.loc[summary["cancer_group"].eq("All cancers")].to_csv(
        output_dir / f"{prefix}_outer_site_performance_summary.csv", index=False)
    frequency.to_csv(output_dir / f"{prefix}_selection_frequency.csv", index=False)
    return dict(per_repeat=per_repeat, summary=summary, frequency=frequency,
                pooled=pooled)


def fit_final_panel(
    cohort: pd.DataFrame, raw_data: dict[str, pd.DataFrame],
    inner_folds: int, seed: int, fractions: tuple[float, ...],
    caps: dict[str, int]
) -> tuple[LassoPanel, dict[str, PreprocessBlock], float]:
    """Tune and fit a deployment candidate using the full modeling cohort.

    Parameters
    ----------
    cohort : pandas.DataFrame
        Patient metadata.
    raw_data : dict[str, pandas.DataFrame]
        Aligned raw predictors.
    inner_folds : int
        Tuning folds.
    seed : int
        Base random seed.
    fractions : tuple[float, ...]
        Candidate lambda fractions.
    caps : dict[str, int]
        Maximum markers per modality.

    Returns
    -------
    tuple
        Capped panel, preprocessing rules, and selected penalty fraction.
    """
    all_idx = np.arange(len(cohort))
    fold_ids = make_stratified_folds(cohort, inner_folds, seed + 9999)
    fraction, _, _ = cv_lasso(cohort, raw_data, all_idx, fold_ids,
                              fractions, seed + 9999)
    prep = fit_model_preprocess(cohort, raw_data, all_idx)
    panel = select_lasso_panel(cohort, raw_data, all_idx, prep,
                               fraction, fractions, seed + 9999, caps)
    return panel, prep, fraction


def annotate_panel(
    panel: LassoPanel, cohort: pd.DataFrame, raw_data: dict[str, pd.DataFrame],
    region_annotation: pd.DataFrame, batch_tests: pd.DataFrame
) -> pd.DataFrame:
    """Attach methylation coordinates, raw means, and batch-effect flags.

    Parameters
    ----------
    panel : LassoPanel
        Final selected coefficients.
    cohort : pandas.DataFrame
        Full modeling cohort with cancer status.
    raw_data : dict[str, pandas.DataFrame]
        Raw assay values aligned to ``cohort``.
    region_annotation : pandas.DataFrame
        Region coordinates and assay-design annotations.
    batch_tests : pandas.DataFrame
        Earlier control-only batch tests.

    Returns
    -------
    pandas.DataFrame
        Annotated final panel, one row per selected marker.
    """
    names = panel.coefficients.index.tolist()
    result = pd.DataFrame({"feature": names,
        "modality": [name.split("__", 1)[0] for name in names],
        "marker_id": [name.split("__", 1)[1] for name in names],
        "standardized_coefficient": panel.coefficients.to_numpy()})
    annotation = region_annotation.rename(columns={"region_id": "marker_id"}).copy()
    annotation["modality"] = "methylation"
    annotation["region_length"] = annotation["end"] - annotation["start"] + 1
    result = result.merge(annotation, on=["modality", "marker_id"], how="left")
    tests = batch_tests.rename(columns={"feature": "marker_id"})
    flags = tests.groupby(["modality", "marker_id"], sort=False).agg(
        has_batch_effect=("significant", "any"),
        batch_effect_variable=("batch_variable", lambda x: ", ".join(
            x[tests.loc[x.index, "significant"]]))).reset_index()
    result = result.merge(flags, on=["modality", "marker_id"], how="left")
    methylation_ids = result.loc[result["modality"].eq("methylation"), "marker_id"]
    if len(methylation_ids):
        raw = raw_data["methylation"][methylation_ids]
        controls = cohort["y"].eq(0).to_numpy()
        cancers = cohort["y"].eq(1).to_numpy()
        means = pd.DataFrame({"marker_id": methylation_ids,
            "mean_methylation_cancer": raw.loc[cancers].mean(skipna=True).to_numpy(),
            "mean_methylation_normal": raw.loc[controls].mean(skipna=True).to_numpy()})
        result = result.merge(means, on="marker_id", how="left")
    return result


def plot_selected_methylation(
    panel_table: pd.DataFrame, cohort: pd.DataFrame,
    raw_methylation: pd.DataFrame
) -> sns.axisgrid.FacetGrid:
    """Boxplot raw selected methylation values by stage, marker, and site.

    Parameters
    ----------
    panel_table : pandas.DataFrame
        Annotated final panel.
    cohort : pandas.DataFrame
        Patient stage and site data.
    raw_methylation : pandas.DataFrame
        Raw methylation values aligned to ``cohort``.

    Returns
    -------
    seaborn.axisgrid.FacetGrid
        Marker-row by site-column boxplots.
    """
    markers = panel_table.loc[panel_table["modality"].eq("methylation"), "marker_id"].tolist()
    values = raw_methylation[markers].copy()
    values["stage_group"] = cohort["stage_group"].to_numpy()
    values["site"] = cohort["site"].to_numpy()
    long = values.melt(id_vars=["stage_group", "site"], var_name="marker_id",
                       value_name="raw_methylation").dropna(subset=["raw_methylation"])
    plot = sns.catplot(data=long, x="stage_group", y="raw_methylation",
        row="marker_id", col="site", kind="box", sharey=False,
        order=["Control", "I", "II", "III", "IV"], height=2, aspect=1.4)
    plot.set_axis_labels("Cancer stage", "Raw methylation")
    return plot


def paired_performance(
    reference: pd.DataFrame, candidate: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare AUC, MCC, and sensitivity on matched repeats and groups.

    Parameters
    ----------
    reference : pandas.DataFrame
        Per-repeat score table for the reference model.
    candidate : pandas.DataFrame
        Per-repeat score table for the candidate model.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame]
        Paired metrics/differences and across-repeat difference summaries.
    """
    keys = ["repeat_id", "site_group", "cancer_group"]
    metrics = ["AUC", "MCC", "sensitivity_at_95_specificity"]
    paired = reference[keys + metrics].merge(candidate[keys + metrics],
                on=keys, how="inner", suffixes=("_molecular", "_fragmentomics"))
    if len(paired) != len(reference):
        raise ValueError("Candidate and reference score groups differ.")
    for metric in metrics:
        paired[f"delta_{metric}"] = (paired[f"{metric}_fragmentomics"] -
                                       paired[f"{metric}_molecular"])
    grouped = paired.groupby(["site_group", "cancer_group"], sort=False)
    summary = grouped.agg(n_repeats=("repeat_id", "nunique"))
    for metric in metrics:
        key = f"delta_{metric}"
        summary[f"{key}_mean"] = grouped[key].mean()
        summary[f"{key}_sd"] = grouped[key].std(ddof=1)
    return paired, summary.reset_index()


def build_fragmentomics_matrix(
    aligned_fragmentomics: pd.DataFrame, cohort: pd.DataFrame
) -> pd.DataFrame:
    """Construct the requested complete fragmentomics predictors.

    Parameters
    ----------
    aligned_fragmentomics : pandas.DataFrame
        QC-filtered fragmentomics table aligned to earliest draws.
    cohort : pandas.DataFrame
        Site A/B protein-assayed modeling patients.

    Returns
    -------
    pandas.DataFrame
        Ratio, mitochondrial fraction, mean length, periodicity, and motifs.
    """
    selected = (aligned_fragmentomics.set_index("sample_id")
                .loc[cohort["sample_id"]].reset_index(drop=True))
    if not np.isfinite(selected["long_frag_fraction"]).all() or \
            (selected["long_frag_fraction"] <= 0).any():
        raise ValueError("Long-fragment fractions must be finite and positive.")
    motifs = [name for name in selected if name.startswith("end_motif_") and
              len(name.split("end_motif_", 1)[1]) == 4 and
              set(name.split("end_motif_", 1)[1]) <= set("ACGT")]
    features = pd.DataFrame({
        "short_to_long_frag_ratio": selected["short_frag_fraction"] /
                                    selected["long_frag_fraction"],
        "mito_frag_fraction": selected["mito_frag_fraction"],
        "mean_frag_length": selected["mean_frag_length"],
        "nucleosome_periodicity": selected["nucleosome_periodicity"]
    })
    features = pd.concat([features, selected[motifs]], axis=1)
    if not np.isfinite(features.to_numpy(dtype=float)).all():
        raise ValueError("Fragmentomics predictors must be complete and finite.")
    return features

In [ ]:
model_cohort, model_raw = prepare_model_inputs(primary_filtered, assays_filtered)
print("Modeling patients:", len(model_cohort),
      "methylation features:", model_raw["methylation"].shape[1],
      "protein features:", model_raw["protein"].shape[1])

molecular_predictions, molecular_selections, molecular_tuning = run_nested_lasso(
    model_cohort, model_raw, REPEATS, OUTER_FOLDS, INNER_FOLDS,
    SEED, LAMBDA_FRACTIONS, MOLECULAR_CAPS
)
molecular_scores = save_model_results(
    "molecular", molecular_predictions, molecular_selections,
    molecular_tuning, OUTPUT_DIR
)
display(molecular_scores["summary"])
display(molecular_scores["frequency"].head(15))

Site-specific results use each site's held-out controls to set an
empirical 95%-specificity cutoff. Each stage is compared with controls from
the same site group. The cutoff is descriptive: it was estimated from the
held-out controls rather than an independent calibration cohort. Stage I has
few cases, and the repeated folds reuse patients.

In [ ]:
stage_plot = molecular_scores["per_repeat"]
stage_plot = stage_plot.loc[stage_plot["site_group"].eq("All sites")]
long_plot = stage_plot.melt(id_vars=["repeat_id", "cancer_group"],
    value_vars=["AUC", "MCC", "sensitivity_at_95_specificity"],
    var_name="metric", value_name="value")
plot = sns.relplot(data=long_plot, x="cancer_group", y="value", hue="repeat_id",
                   col="metric", kind="line", marker="o", facet_kws={"sharey": False},
                   height=3.3, aspect=1.15)
plot.set_axis_labels("Cancer stage", "Metric value")
plt.show()

The above results show that the model performs better in late-stage cancers than early ones.

The full-cohort fit below is intended as a future-use panel; its own
training performance is not a held-out estimate. Raw methylation means and
region annotations help assess biological patterns and assay feasibility.

In [ ]:
molecular_panel, molecular_prep, molecular_fraction = fit_final_panel(
    model_cohort, model_raw, INNER_FOLDS, SEED, LAMBDA_FRACTIONS, MOLECULAR_CAPS
)
molecular_panel_table = annotate_panel(
    molecular_panel, model_cohort, model_raw, tables["region_annotation"],
    batch_effect_tests
)
molecular_panel_table.to_csv(OUTPUT_DIR / "molecular_final_panel.csv", index=False)
display(molecular_panel_table)

We also plot the methylation values for these markers across the cancer stages, from which we can see that the methylation goes up as cancer stages progress; but the pattern is a bit different between site A and B, where the site B stage I cancers tend to have higher methylation values than stage II, suggesting sample heterogeneity.

In [ ]:
plot_selected_methylation(molecular_panel_table, model_cohort, model_raw["methylation"])
plt.show()

The original panel contains five methylation and three protein markers.
All selected methylation regions either overlap repeats or have high GC
content. These properties may complicate primer and probe design for ddPCR.

### Refit after repeat and GC filtering

We remove methylation regions overlapping repeats and regions with GC content
outside [0.4, 0.6]. The same patients, protein data, seeds, and outer fold
construction are retained, allowing a paired comparison of held-out AUCs.

In [ ]:
region_eligibility = tables["region_annotation"].copy()
region_eligibility["eligible"] = (
    region_eligibility["repeat_overlap"].notna() &
    ~region_eligibility["repeat_overlap"].fillna(True).astype(bool) &
    region_eligibility["gc_content"].between(0.4, 0.6)
)
region_eligibility.to_csv(OUTPUT_DIR / "feasible_methylation_eligibility.csv", index=False)
eligible_ids = [feature for feature in model_raw["methylation"]
                if feature in set(region_eligibility.loc[region_eligibility["eligible"], "region_id"])]
feasible_raw = {**model_raw, "methylation": model_raw["methylation"][eligible_ids]}
print("Assay-design-eligible methylation regions:", len(eligible_ids))

feasible_predictions, feasible_selections, feasible_tuning = run_nested_lasso(
    model_cohort, feasible_raw, REPEATS, OUTER_FOLDS, INNER_FOLDS,
    SEED, LAMBDA_FRACTIONS, MOLECULAR_CAPS
)
feasible_scores = save_model_results(
    "feasible_molecular", feasible_predictions, feasible_selections,
    feasible_tuning, OUTPUT_DIR
)
display(feasible_scores["summary"])
display(feasible_scores["frequency"].head(15))
feasible_auc_comparison = molecular_scores["pooled"][["repeat_id", "AUC"]].rename(
    columns={"AUC": "AUC_original"}
).merge(feasible_scores["pooled"][["repeat_id", "AUC"]].rename(
    columns={"AUC": "AUC_feasible"}), on="repeat_id")
feasible_auc_comparison["AUC_difference"] = (
    feasible_auc_comparison["AUC_feasible"] - feasible_auc_comparison["AUC_original"]
)
feasible_auc_comparison.to_csv(OUTPUT_DIR / "feasible_molecular_auc_comparison.csv", index=False)
display(feasible_auc_comparison)

feasible_panel, feasible_prep, feasible_fraction = fit_final_panel(
    model_cohort, feasible_raw, INNER_FOLDS, SEED, LAMBDA_FRACTIONS, MOLECULAR_CAPS
)
feasible_panel_table = annotate_panel(
    feasible_panel, model_cohort, feasible_raw, tables["region_annotation"],
    batch_effect_tests
)
feasible_panel_table.to_csv(OUTPUT_DIR / "feasible_molecular_final_panel.csv", index=False)
display(feasible_panel_table)

Filtering leaves 848 of 2,000 regions. The paired
AUC table above shows the change in values before and after removing design-infeasible methylation markers. A decrease can reflect
the loss of predictive regions; assay feasibility still needs experimental
evaluation.

### Add fragmentomics features

The new inputs are the short-to-long fragment ratio, mitochondrial fragment
fraction, mean fragment length, nucleosome periodicity, and each measured
4-mer end-motif fraction. The fragmentomics features have no missing values
or detected batch effects; only fold-specific standardization is applied.
LASSO retains at most two fragmentomics features alongside the five
methylation and three protein features. This comparison uses the **original**
methylation + protein inputs, without repeat or GC filtering.

In [ ]:
fragmentomics_matrix = build_fragmentomics_matrix(
    assays_filtered["fragmentomics"], model_cohort
)
fragmentomics_raw = {**model_raw, "fragmentomics": fragmentomics_matrix}
display(pd.DataFrame({"modality": list(fragmentomics_raw),
                      "n_features": [x.shape[1] for x in fragmentomics_raw.values()]}))

fragmentomics_predictions, fragmentomics_selections, fragmentomics_tuning = run_nested_lasso(
    model_cohort, fragmentomics_raw, REPEATS, OUTER_FOLDS, INNER_FOLDS,
    SEED, LAMBDA_FRACTIONS, FRAGMENTOMICS_CAPS
)
reference_keys = molecular_predictions[["repeat_id", "fold", "patient_id"]]
candidate_keys = fragmentomics_predictions[["repeat_id", "fold", "patient_id"]]
if not reference_keys.equals(candidate_keys):
    raise ValueError("The molecular and fragmentomics outer folds differ.")
fragmentomics_scores = save_model_results(
    "fragmentomics", fragmentomics_predictions, fragmentomics_selections,
    fragmentomics_tuning, OUTPUT_DIR
)
display(fragmentomics_scores["summary"])
display(fragmentomics_scores["frequency"].head(15))
fragmentomics_comparison, fragmentomics_comparison_summary = paired_performance(
    molecular_scores["per_repeat"], fragmentomics_scores["per_repeat"]
)
fragmentomics_comparison.to_csv(OUTPUT_DIR / "fragmentomics_performance_comparison.csv", index=False)
fragmentomics_comparison_summary.to_csv(
    OUTPUT_DIR / "fragmentomics_performance_comparison_summary.csv", index=False
)
display(fragmentomics_comparison_summary.loc[
    fragmentomics_comparison_summary["cancer_group"].eq("All cancers")
])

fragmentomics_panel, fragmentomics_prep, fragmentomics_fraction = fit_final_panel(
    model_cohort, fragmentomics_raw, INNER_FOLDS, SEED,
    LAMBDA_FRACTIONS, FRAGMENTOMICS_CAPS
)
fragmentomics_panel_table = annotate_panel(
    fragmentomics_panel, model_cohort, fragmentomics_raw,
    tables["region_annotation"], batch_effect_tests
)
fragmentomics_panel_table.to_csv(OUTPUT_DIR / "fragmentomics_final_panel.csv", index=False)
display(fragmentomics_panel_table)

The results show that fragmentomics add little to the model performance in terms of AUC and sensitivity, interesting the feature short_to_long_frag_ratio are frequently show up in models during cross validation, suggesting its robustness.

### Generated files

The notebook saves the following CSV files in `./report_artifacts`.
**Outer** results come from held-out patients in repeated nested
cross-validation. **Final panel** coefficients come from a fit to the full
modeling cohort.

#### Quality control and batch effects

- `assay_coverage.csv` — number of primary samples with an assay row for each modality.
- `feature_missing.csv` — missing-value counts and fractions for each feature.
- `sample_missing.csv` — missing-value counts and fractions for each sample and modality.
- `batch_effect_tests.csv` — control-only site and plate effect tests, with raw and adjusted p-values.
- `batch_effect_coefficients.csv` — fitted coefficients from the control-only batch models.
- `batch_adjusted_values.csv` — original and batch-adjusted measurements for affected features, with the estimated effects and coefficients.

#### Original methylation + protein model

This model uses all QC-passing methylation regions.

- `molecular_outer_predictions.csv` — held-out cancer probabilities and patient, site, stage, and fold identifiers.
- `molecular_outer_selections.csv` — markers and coefficients selected in each outer training fold.
- `molecular_outer_tuning.csv` — inner-CV penalty choice, log loss, fold sizes, and selected marker counts.
- `molecular_outer_performance.csv` — all-cancer metrics pooled across sites, separately for each repeat.
- `molecular_outer_stage_performance.csv` — all-cancer and stage-specific metrics for pooled sites and each site, by repeat.
- `molecular_outer_site_performance.csv` — all-cancer metrics for pooled sites and each site, by repeat.
- `molecular_outer_performance_summary.csv` — mean and SD across repeats for every site and cancer-stage group.
- `molecular_outer_site_performance_summary.csv` — mean and SD across repeats for all cancers at pooled and individual sites.
- `molecular_selection_frequency.csv` — number and fraction of outer folds selecting each marker.
- `molecular_final_panel.csv` — full-cohort panel coefficients, raw cancer/control methylation means, methylation annotations, and batch-effect flags.

#### Methylation + protein model with assay-design filters

This model excludes methylation regions overlapping repeats or outside the
GC-content range [0.4, 0.6].

- `feasible_methylation_eligibility.csv` — region annotations and whether each region passed both filters.
- `feasible_molecular_outer_predictions.csv` — held-out cancer probabilities and patient, site, stage, and fold identifiers.
- `feasible_molecular_outer_selections.csv` — markers and coefficients selected in each outer training fold.
- `feasible_molecular_outer_tuning.csv` — inner-CV penalty choice, log loss, fold sizes, and selected marker counts.
- `feasible_molecular_outer_performance.csv` — all-cancer metrics pooled across sites, separately for each repeat.
- `feasible_molecular_outer_stage_performance.csv` — all-cancer and stage-specific metrics for pooled sites and each site, by repeat.
- `feasible_molecular_outer_site_performance.csv` — all-cancer metrics for pooled sites and each site, by repeat.
- `feasible_molecular_outer_performance_summary.csv` — mean and SD across repeats for every site and cancer-stage group.
- `feasible_molecular_outer_site_performance_summary.csv` — mean and SD across repeats for all cancers at pooled and individual sites.
- `feasible_molecular_selection_frequency.csv` — number and fraction of outer folds selecting each marker.
- `feasible_molecular_auc_comparison.csv` — paired all-cancer AUCs by repeat for the original and filtered models.
- `feasible_molecular_final_panel.csv` — full-cohort filtered panel coefficients, methylation annotations, and batch-effect flags.

#### Methylation + protein + fragmentomics model

This model uses all QC-passing methylation regions and adds the selected
fragmentomics inputs. Its comparison files use the original methylation +
protein model as the reference.

- `fragmentomics_outer_predictions.csv` — held-out cancer probabilities and patient, site, stage, and fold identifiers.
- `fragmentomics_outer_selections.csv` — markers and coefficients selected in each outer training fold.
- `fragmentomics_outer_tuning.csv` — inner-CV penalty choice, log loss, fold sizes, and selected marker counts.
- `fragmentomics_outer_performance.csv` — all-cancer metrics pooled across sites, separately for each repeat.
- `fragmentomics_outer_stage_performance.csv` — all-cancer and stage-specific metrics for pooled sites and each site, by repeat.
- `fragmentomics_outer_site_performance.csv` — all-cancer metrics for pooled sites and each site, by repeat.
- `fragmentomics_outer_performance_summary.csv` — mean and SD across repeats for every site and cancer-stage group.
- `fragmentomics_outer_site_performance_summary.csv` — mean and SD across repeats for all cancers at pooled and individual sites.
- `fragmentomics_selection_frequency.csv` — number and fraction of outer folds selecting each marker.
- `fragmentomics_performance_comparison.csv` — paired AUC, MCC, and sensitivity differences versus the original model for each repeat, site, and cancer group.
- `fragmentomics_performance_comparison_summary.csv` — mean and SD of those paired differences across repeats.
- `fragmentomics_final_panel.csv` — full-cohort panel coefficients, methylation annotations, and batch-effect flags where applicable.

## Interpretation and next steps

The following summary is computed from this Python run. The R report found
that filtering methylation regions reduced discrimination and that the tested
fragmentomics features did not improve the original model. Exact Python
numbers may differ because the solvers and fold assignments differ.

In [ ]:
def pooled_metric(summary: pd.DataFrame, metric: str) -> float:
    """Return a pooled all-cancer mean from a score summary.

    Parameters
    ----------
    summary : pandas.DataFrame
        Repeated prediction summary.
    metric : str
        Name of the mean metric column.

    Returns
    -------
    float
        Pooled all-cancer metric.
    """
    row = summary.loc[summary["site_group"].eq("All sites") &
                      summary["cancer_group"].eq("All cancers")]
    return float(row.iloc[0][metric])

base_auc = pooled_metric(molecular_scores["summary"], "AUC_mean")
filtered_auc = pooled_metric(feasible_scores["summary"], "AUC_mean")
frag_auc = pooled_metric(fragmentomics_scores["summary"], "AUC_mean")
base_sens = pooled_metric(molecular_scores["summary"], "sensitivity_at_95_specificity_mean")
filtered_sens = pooled_metric(feasible_scores["summary"], "sensitivity_at_95_specificity_mean")
frag_markers = fragmentomics_panel_table.loc[
    fragmentomics_panel_table["modality"].eq("fragmentomics"), "marker_id"
].tolist()
display(Markdown(
    f"**Pooled held-out AUC:** original {base_auc:.3f}; filtered methylation "
    f"{filtered_auc:.3f}; with fragmentomics {frag_auc:.3f}. "
    f"**Sensitivity at 95% specificity:** original {base_sens:.3f}; "
    f"filtered {filtered_sens:.3f}. The final fragmentomics panel selected "
    f"{', '.join(frag_markers) if frag_markers else 'no fragmentomics feature'}."
))
print("Saved CSV files:", len(list(OUTPUT_DIR.glob("*.csv"))))
# print("Artifact directory:", OUTPUT_DIR)

The repeated folds reuse patients, so their across-repeat SDs describe
split variation rather than independent-cohort uncertainty. Sensitivity and
MCC at 95% specificity use cutoffs estimated from held-out controls; a
clinical threshold should be fixed before external testing. Only sites A and
B enter protein-based models, and stage I has few cases.

- Fix the panel and preprocessing rules, then validate the model in an
  independent cohort using suitable public or internal data. Assess how well
  it generalizes across populations, sites, and assay runs.
- Collect more stage I and II cases and revisit model development with adequate
  early-stage representation. Evaluate additional methylation regions, proteins,
  and fragmentomics features for incremental predictive value.
- Check candidate methylation regions for overlap with known polymorphisms,
  since variants in primer or probe binding sites could affect assay performance.
- Work with laboratory scientists to assess whether primers and probes can be
  designed reliably for the high-GC methylation regions, and test the resulting
  assays experimentally.
- Compare early- and late-stage cancers to identify candidate markers of
  progression, then evaluate those markers in longitudinal samples before
  using them to monitor disease over time.
